# Speaker Inference -> Prompt Testing

Tests the prompt that identifies real names from a transcript with speaker IDs.

**Input**: transcript with `Speaker 0`, `Speaker 1`, etc. + list of speaker IDs
**Output**: JSON mapping each speaker ID to a real name with confidence + evidence quote

The prompt lives in `prompts/deepgram/speaker_inference.yaml` under the key `system`.

In [ ]:
import sys, json, time, copy, requests
from pathlib import Path
from datetime import datetime

sys.path.insert(0, "..")
from utils.api_client import call_model
from utils.prompt_loader import load_prompt

PROMPTS = load_prompt("../prompts/deepgram/speaker_inference.yaml")
print(f"Loaded speaker_inference.yaml -- system prompt: {len(PROMPTS['system'])} chars")

## 0. Production Logs (Optional)

Fetch speaker inference logs from production to see what the current prompt does.

In [ ]:
WORKFLOW_URL = "https://workflow-service.mangowater-31feba87.swedencentral.azurecontainerapps.io"

# Fetch all logs and filter for speaker inference
response = requests.get(f"{WORKFLOW_URL}/debug/prompt-logs", params={"limit": 100}, timeout=10)

if response.status_code != 200:
    print(f"Error {response.status_code} — skip to Section 1")
else:
    all_logs = response.json()
    # Filter for speaker inference by checking the prompt content
    logs = [l for l in all_logs if "forensic transcript analyst" in l.get("prompt_preview", "").lower()
            or "speaker" in l.get("pass", "").lower()]

    if not logs:
        print("No speaker inference logs yet. Run a meeting first, or skip to Section 1.")
    else:
        print(f"Found {len(logs)} speaker inference entries\n")
        for i, entry in enumerate(logs):
            print(f"[{i}] {entry['pass']}  |  {entry['model']}  |  {entry['prompt_length']} chars  |  {entry['timestamp']}")
            print(f"     {entry['prompt_preview'][:120]}...")
            print()

In [ ]:
# Pick one to inspect
LOG_INDEX = 0

entry = logs[LOG_INDEX]
print(f"Pass: {entry['pass']}  |  Model: {entry['model']}  |  {entry['timestamp']}")
print(f"Prompt: {entry['prompt_length']} chars  |  Response: {entry['response_length']} chars")
print(f"\n=== PRODUCTION PROMPT ===")
print(entry["prompt_preview"][:2000])
print(f"\n=== PRODUCTION OUTPUT ===")
print(entry["response_preview"])

## 1. Load Transcript + Speaker IDs

The transcript must have `Speaker 0`, `Speaker 1` (or `Sprecher 0`, etc.) labels.
List the speaker IDs you want the model to identify.

In [ ]:
# --- CONFIGURE THESE ---
TRANSCRIPT_FILE = "../test_cases/meeting_01.txt"
SPEAKER_IDS = [0, 1]          # which speaker IDs to identify
MODEL = "gpt-4o"              # "gpt-4o" or "mistral-25b"

transcript = Path(TRANSCRIPT_FILE).read_text(encoding="utf-8")
print(f"{TRANSCRIPT_FILE}  |  {len(transcript)} chars  |  {MODEL}")
print(f"Speaker IDs to identify: {SPEAKER_IDS}")
print("---")
print(transcript[:500] + "..." if len(transcript) > 500 else transcript)

## 2. Run the Prompt (Baseline)

In [ ]:
def build_user_message(speaker_ids, transcript):
    """Build the user message: speaker IDs to identify + transcript."""
    return f"Speaker IDs to identify: {speaker_ids}\n\nTRANSCRIPT:\n{transcript}"


system_prompt = PROMPTS["system"]
user_prompt = build_user_message(SPEAKER_IDS, transcript)

print(f"System prompt: {len(system_prompt)} chars")
print(f"User prompt: {len(user_prompt)} chars")
print(f"\n=== SYSTEM PROMPT ===")
print(system_prompt)

In [ ]:
print(f"Running {MODEL}...")
start = time.time()
baseline_output = call_model(system_prompt, user_prompt, model=MODEL, max_tokens=2000)
baseline_duration = round(time.time() - start, 2)
print(f"Done in {baseline_duration}s\n")

try:
    clean = baseline_output.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1].rsplit("```", 1)[0]
    baseline_result = json.loads(clean)
    print(json.dumps(baseline_result, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print(baseline_output)
    baseline_result = None

## 3. Edit the Prompt and Re-run

In [ ]:
# See the current prompt
print(PROMPTS["system"])

In [ ]:
# Make a copy and override the prompt
PROMPTS_MODIFIED = copy.deepcopy(PROMPTS)

# Uncomment and paste your modified version:

# PROMPTS_MODIFIED["system"] = """
# ... your changes ...
# """

if PROMPTS["system"] != PROMPTS_MODIFIED["system"]:
    print(f"Changed: system prompt")
    print(f"  ({len(PROMPTS['system'])} -> {len(PROMPTS_MODIFIED['system'])} chars)")
else:
    print("Nothing changed yet -- uncomment and paste your edits above.")

In [ ]:
# Run with the modified prompt
modified_system = PROMPTS_MODIFIED["system"]

print(f"Running {MODEL} with modified prompt...")
start = time.time()
modified_output = call_model(modified_system, user_prompt, model=MODEL, max_tokens=2000)
modified_duration = round(time.time() - start, 2)

try:
    clean = modified_output.strip()
    if clean.startswith("```"):
        clean = clean.split("\n", 1)[1].rsplit("```", 1)[0]
    modified_result = json.loads(clean)
    print(f"Done in {modified_duration}s\n")
    print(json.dumps(modified_result, indent=2, ensure_ascii=False))
except json.JSONDecodeError as e:
    print(f"JSON parse error: {e}")
    print(modified_output)
    modified_result = None

## 4. Compare

In [ ]:
print("=" * 80)
print(f"BASELINE -- {baseline_duration}s")
print("=" * 80)
if baseline_result:
    print(json.dumps(baseline_result, indent=2, ensure_ascii=False))
else:
    print(baseline_output)

print(f"\n{'=' * 80}")
print(f"MODIFIED -- {modified_duration}s")
print("=" * 80)
if modified_result:
    print(json.dumps(modified_result, indent=2, ensure_ascii=False))
else:
    print(modified_output)

## 5. Save Results

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
name = Path(TRANSCRIPT_FILE).stem

results = {
    "transcript": name,
    "speaker_ids": SPEAKER_IDS,
    "model": MODEL,
    "timestamp": timestamp,
    "baseline": {
        "output": baseline_output,
        "result": baseline_result,
        "duration": baseline_duration,
    },
    "modified": {
        "output": modified_output,
        "result": modified_result,
        "duration": modified_duration,
    },
}

outpath = Path(f"../results/speaker_{name}_{timestamp}.json")
outpath.parent.mkdir(parents=True, exist_ok=True)
outpath.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved to {outpath}")